# Tarea 1 - Predicción de resultados del fútbol uruguayo

Este notebook es nuestra solución y documentación de la tarea 1 del curso de Aprendizaje Automático 2026.

## 1. Objetivo y criterios de evaluación

## 1. Objetivo y criterios de evaluación

### Objetivo

El objetivo de esta tarea es predecir el resultado de un partido de fútbol uruguayo antes de que se juegue.

La variable objetivo se llama `winner` y puede tomar tres valores:

- `L`: gana el equipo local.
- `V`: gana el equipo visitante.
- `E`: el partido termina empatado.

Para realizar la predicción se utilizarán únicamente datos disponibles antes de cada partido. En particular, por ejemplo, los goles de cada equipo (`gh` y `ga`) se usarán solamente para crear la variable objetivo, pero nunca como atributos de entrada del modelo.

### Conjuntos de datos

Los partidos hasta el año 2023 inclusive se utilizarán para entrenamiento y ajuste de hiperparámetros mediante validación cruzada.

Los partidos de 2024 y 2025 se reservarán exclusivamente para la evaluación final. No se usarán para ajustar atributos ni hiperparámetros.

### Modelos a comparar

Se implementarán los siguientes clasificadores:

1. Clasificador base para las comparaciones, basado en la proporción de victorias de cada equipo en los últimos diez años.
2. Árbol de decisión propio, usando el hiperparámetro `min_info_gain` que es el mínimo de ganancia de información que debe haber para que el árbol siga creciendo.
3. Naive Bayes propio, usando el hiperparámetro `m` que representa cuántos ejemplos “imaginarios” basados en la distribución general agregamos para cada clase.
4. Random Forest de scikit-learn.
5. Naive Bayes de scikit-learn.

### Métricas de evaluación

Los modelos se compararán utilizando:

- Accuracy.
- Precision por clase.
- Recall por clase.
- F1-score por clase.
- Macro-F1.
- Matriz de confusión.

Se utilizará validación cruzada temporal para seleccionar los hiperparámetros, evitando usar información de partidos futuros.

## 2. Importaciones y configuración

Importamos pandas, numpy, matplotlib y scikit-learn. Definimos `RANDOM_STATE = 50` para que los experimentos sean reproducibles y constantes.

In [14]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import sklearn as sl

RANDOM_STATE = 50
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Configuración cargada.")
print("Semilla aleatoria:", RANDOM_STATE)

Configuración cargada.
Semilla aleatoria: 50


## 3. Carga e inspección inicial del dataset

Leemos `futbol_uruguayo.csv` utilizando DataFrame de Pandas. Verificamos cantidad de filas y columnas, primeras filas, tipos de datos, valores faltantes, duplicados y rango de fechas. Por último, convertimos `date` de String al tipo fecha de Pandas, datetime.

In [ ]:
df_original = pd.read_csv("futbol_uruguayo/futbol_uruguayo.csv")

print("Filas y columnas:", df_original.shape)
print("\nColumnas:")
print(df_original.columns.tolist())

display(df_original.head())

Filas y columnas: (15207, 17)

Columnas:
['home', 'away', 'date', 'gh', 'ga', 'full_time', 'competition', 'home_ident', 'away_ident', 'home_country', 'away_country', 'home_code', 'away_code', 'home_continent', 'away_continent', 'continent', 'level']


,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national


In [ ]:
print("Tipos de datos:")
display(df_original.dtypes)

print("\nInformación general:\n")
df_original.info()

Tipos de datos:


home                  str
away                  str
date                  str
gh                float64
ga                float64
full_time             str
competition           str
home_ident            str
away_ident            str
home_country          str
away_country          str
home_code             str
away_code             str
home_continent        str
away_continent        str
continent             str
level                 str
dtype: object


Información general:

<class 'pandas.DataFrame'>
RangeIndex: 15207 entries, 0 to 15206
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   home            15207 non-null  str    
 1   away            15207 non-null  str    
 2   date            15207 non-null  str    
 3   gh              15207 non-null  float64
 4   ga              15207 non-null  float64
 5   full_time       15207 non-null  str    
 6   competition     15207 non-null  str    
 7   home_ident      15207 non-null  str    
 8   away_ident      15207 non-null  str    
 9   home_country    15207 non-null  str    
 10  away_country    15207 non-null  str    
 11  home_code       15207 non-null  str    
 12  away_code       15207 non-null  str    
 13  home_continent  15207 non-null  str    
 14  away_continent  15207 non-null  str    
 15  continent       15207 non-null  str    
 16  level           15207 non-null  str    
dtypes: float64(2), str(

In [ ]:
print("Valores faltantes:")
display(df_original.isna().sum())

print("\nDuplicados exactos:\n")
print(df_original.duplicated().sum())

Valores faltantes:


home              0
away              0
date              0
gh                0
ga                0
full_time         0
competition       0
home_ident        0
away_ident        0
home_country      0
away_country      0
home_code         0
away_code         0
home_continent    0
away_continent    0
continent         0
level             0
dtype: int64


Duplicados exactos:

1


In [ ]:
df_original["date"] = pd.to_datetime(df_original["date"], errors="coerce")

print("Fechas inválidas:", df_original["date"].isna().sum())
print("Fecha mínima y máxima:", df_original["date"].min(), ",", df_original["date"].max())

print("Cantidad de equipos visitantes distintos:", df_original["away"].nunique())
print("Cantidad de equipos locales distintos:", df_original["home"].nunique())

Fechas inválidas: 0
Fecha mínima y máxima: 1932-03-05 00:00:00 , 2025-06-30 00:00:00
Cantidad de equipos visitantes distintos: 38
Cantidad de equipos locales distintos: 38


## 4. Limpieza y creación de la variable objetivo `winner`

Creamos el objetivo `winner`: `L` si `gh > ga`, `V` si `gh < ga` y `E` si son iguales. Los goles solo se usan para crear esta variable, no para predecir.
Para la predicción final borramos todas las columnas menos home, away y date. La columna full_time la eliminamos ya que describe cómo terminó el partido y, por lo tanto, no se conoce antes de jugarlo. Usarla podría introducir fuga de información. Además no aportaba informacion relevante para clasificar el ganador.

In [ ]:
df = df_original.copy()

df["winner"] = df.apply(
    lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"),
    axis=1
)

print("Dataset con variable objetivo:")
print(df)

## 5. Análisis exploratorio de los datos

Analiza cantidad de partidos por año y distribución de `winner`. Incluye gráficas simples y escribe qué observas, por ejemplo si hay más victorias locales que empates.

## 6. Creación de atributos disponibles antes del partido

Ordenamos por fecha y construímos estadísticas usando solo partidos anteriores: rendimiento reciente, proporción de victorias, localía, visitante, diferencias entre equipos, nunca usando resultados futuros.

## 7. División temporal: entrenamiento y evaluación

Separa entrenamiento hasta 2023 inclusive y test final con 2024-2025. Muestra tamaño y distribución de clases de cada conjunto. No uses una división aleatoria para el test final.

## 8. Clasificador base de últimos diez años

Implementa el baseline exigido: predice que gana el equipo con mayor proporción general de victorias durante los diez años anteriores al partido. Define qué harás si ambas proporciones empatan.

## 9. Preprocesamiento reproducible

Construye pipelines de scikit-learn para imputar faltantes, transformar atributos numéricos y codificar atributos categóricos. El pipeline se ajusta solo con entrenamiento en cada fold.

## 10. Árbol de decisión propio

Implementa entropía, ganancia de información, elección de división y hojas. Incluye el hiperparámetro `min_info_gain`, que detiene la recursión cuando ninguna ganancia lo supera.

## 11. Naive Bayes propio

Implementa probabilidades a priori y condicionales. Usa el hiperparámetro `m` para suavizado y prueba distintos valores.

## 12. Modelos de scikit-learn

Entrena Random Forest y Naive Bayes de scikit-learn con los datos preprocesados. Fija la semilla de Random Forest.

## 13. Validación cruzada temporal e hiperparámetros

Usa solo partidos hasta 2023 y una estrategia temporal, como `TimeSeriesSplit`. Ajusta `min_info_gain`, `m` y los parámetros de los modelos de scikit-learn sin usar el test final.

## 14. Gráficas de resultados

Grafica el error o `1 - macro-F1` al cambiar `min_info_gain` y `m`. Guarda las gráficas finales en `resultados/` si también las usarás en el informe.

## 15. Evaluación final sobre 2024-2025

Entrena cada modelo elegido con todos los partidos hasta 2023. Evalúa una sola vez en 2024-2025 y presenta accuracy, precision, recall, F1 por clase, macro-F1 y matriz de confusión.

## 16. Análisis de errores y conclusiones

Compara los modelos. Explica cuál funcionó mejor, si los empates fueron difíciles de detectar y en qué tipos de partidos un modelo tiende a equivocarse. Indica limitaciones de los datos.

## 17. Ejemplos de predicción

Al final, agrega una función que reciba un partido futuro y devuelva una predicción. Muestra dos o tres ejemplos de uso con el mejor modelo.